In [38]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import pandas as pd
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
from scipy.spatial.distance import cdist
import xarray as xr

import sys
sys.path.append('..')
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

def gaussian_model(x, mu, sigma):
    y = np.exp(-0.5*(x-mu)**2 / sigma**2)
    return y/np.max(y)

In [ ]:
Overall_XR = xr.DataArray(data=np.zeros([len(cameras), len(dyes), len(n_photon_space), len(parameters)]),
                         coords=[labels, dyes, n_photon_space, parameters],
                         dims=['Camera_Type', 'Dye', 'Photon', 'Analysis_Parameter'])
for i, camera in enumerate(cameras):
    loop_files = np.sort([x for x in all_files if camera in x])
    for j, dye in enumerate(dyes):
        dye_file = np.sort([os.path.join(save_folder_refactored, x) for x in loop_files if dye in x])
        dye_file_raw = np.sort([x for x in dye_file if "rawresults" in x])
        dye_xy_gt = pd.read_csv(np.sort([x for x in dye_file if "groundtruth" in x])[0])
        dye_x0 = dye_xy_gt['x0'].to_numpy()/69
        dye_y0 = dye_xy_gt['y0'].to_numpy()/69
        x0y0 = np.vstack([
                dye_x0,
                dye_y0, 
            ]).T
        dye_BGR = pd.read_csv(np.sort([x for x in dye_file if "input_parameters" in x])[0]).to_numpy()[0][-3:]
        for k, photonval in enumerate(n_photon_space):
            results = pd.read_csv(dye_file_raw[k])
            XY = np.vstack([
                results['x0'].to_numpy(),
                results['y0'].to_numpy(), 
            ]).T
            spatial_distances = euclidean_distance(x0y0, XY)
            colour_loc = np.expand_dims(dye_BGR, 0)
            colour = np.vstack([
                results['A_B'].to_numpy(),
                results['A_G'].to_numpy(), 
                results['A_R'].to_numpy()
            ]).T
            colour_distances = cdist(colour, colour_loc)
            Overall_XR[i, j, k, 0] = np.nanstd(spatial_distances)*69
            Overall_XR[i, j, k, 1] = np.nanstd(colour_distances)

Overall_XR.to_netcdf(os.path.join(save_folder_refactored, '3Camera_Database.nc'))